# Retrieving the data

In [2]:
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

fig_dir = "/Users/alberto/Documents/projects/GWP_1/RiskManagement/GWP1/figures"
data_dir = "/Users/alberto/Documents/projects/GWP_1/RiskManagement/GWP1/data"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

# 1. WTI Crude Oil (Yahoo)
oil = yf.download("CL=F", start="2010-01-01", end=datetime.today().strftime('%Y-%m-%d'), progress=False)
oil.to_csv(os.path.join(data_dir, "oil_prices.csv"))
print("Oil data shape:", oil.shape)
oil['Close'].plot(figsize=(10,6), title="WTI Crude Oil Futures Price")
plt.savefig(os.path.join(fig_dir, "oil_price_time_series.png"))
plt.close()

# 2. Alternative Macro Data via pandas_datareader (FRED)
start = "2010-01-01"
try:
    macro = web.DataReader(['CPIAUCSL', 'FEDFUNDS', 'INDPRO', 'DCOILWTICO'], 'fred', start)
    macro.to_csv(os.path.join(data_dir, "fred_macro.csv"))
    print("Macro data shape:", macro.shape)
    print(macro.tail())
except Exception as e:
    print("pandas_datareader error:", e)
    # Fallback: manual CSV if needed

# 3. Simple EIA example (monthly WTI spot via direct link or expand later)
print("\nData saved. Run EDA next.")

# Quick correlation plot (once more data available)
if not macro.empty:
    sns.heatmap(macro.corr(), annot=True, cmap='coolwarm')
    plt.title("Macro Variables Correlation")
    plt.savefig(os.path.join(fig_dir, "macro_correlation.png"))
    plt.close()

Oil data shape: (4156, 5)
Macro data shape: (4368, 4)
            CPIAUCSL  FEDFUNDS  INDPRO  DCOILWTICO
DATE                                              
2026-07-07       NaN       NaN     NaN       71.53
2026-07-08       NaN       NaN     NaN       74.56
2026-07-09       NaN       NaN     NaN       73.15
2026-07-10       NaN       NaN     NaN       72.45
2026-07-13       NaN       NaN     NaN       79.20

Data saved. Run EDA next.


/var/folders/54/tt8_6d357691t8wwz7yxv9880000gn/T/ipykernel_20693/593300448.py:25: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  macro = web.DataReader(['CPIAUCSL', 'FEDFUNDS', 'INDPRO', 'DCOILWTICO'], 'fred', start)


# Cleaning

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

fig_dir = "/Users/alberto/Documents/projects/GWP_1/RiskManagement/GWP1/figures"
data_dir = "/Users/alberto/Documents/projects/GWP_1/RiskManagement/GWP1/data"

# Load oil (already fixed from your output)
oil = pd.read_csv(os.path.join(data_dir, "oil_prices.csv"), 
                  skiprows=3,
                  names=['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'],
                  parse_dates=['Date'], 
                  index_col='Date')

macro = pd.read_csv(os.path.join(data_dir, "fred_macro.csv"), 
                    parse_dates=['DATE'], index_col='DATE')

print("Oil shape:", oil.shape)
print("Macro shape:", macro.shape)

# Combine - use 'ME' instead of 'M'
df = pd.concat([
    oil['Close'].resample('ME').last().rename('WTI_Close'),
    macro.resample('ME').last()
], axis=1)

print("\nCombined shape before cleaning:", df.shape)

# Cleaning
def remove_outliers(series, factor=3.0):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return series.clip(Q1 - factor*IQR, Q3 + factor*IQR)

for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = remove_outliers(df[col])

df = df[~df.index.duplicated(keep='first')].sort_index()
df = df.interpolate(method='time').ffill().bfill()

print("Missing after cleaning:", df.isna().sum().sum())
print("Final cleaned shape:", df.shape)

df.to_csv(os.path.join(data_dir, "cleaned_oil_macro.csv"))

# Plots
plt.figure(figsize=(12,6))
df['WTI_Close'].plot(title="Cleaned Monthly WTI Crude Oil Price")
plt.savefig(os.path.join(fig_dir, "cleaned_wti_price.png"))
plt.close()

plt.figure(figsize=(10,6))
sns.boxplot(data=df)
plt.title("Distributions After Cleaning")
plt.xticks(rotation=45)
plt.savefig(os.path.join(fig_dir, "post_cleaning_boxplot.png"))
plt.close()

print("✅ Cleaning completed!")

Oil shape: (4156, 6)
Macro shape: (4368, 4)

Combined shape before cleaning: (199, 5)
Missing after cleaning: 0
Final cleaned shape: (199, 5)
✅ Cleaning completed!


# Exploratory data analysis

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

fig_dir = "/Users/alberto/Documents/projects/GWP_1/RiskManagement/GWP1/figures"
data_dir = "/Users/alberto/Documents/projects/GWP_1/RiskManagement/GWP1/data"

# Load without assuming column name
df = pd.read_csv(os.path.join(data_dir, "cleaned_oil_macro.csv"), 
                 parse_dates=[0], 
                 index_col=0)

print("Loaded columns:", df.columns.tolist())
print(df.head())

# Returns
df['WTI_Returns'] = df['WTI_Close'].pct_change() * 100

# Plots (same as before)
plt.figure(figsize=(12,5))
sns.histplot(df['WTI_Returns'].dropna(), kde=True)
plt.title("Distribution of Oil Returns")
plt.savefig(os.path.join(fig_dir, "returns_distribution.png"))
plt.close()

plt.figure(figsize=(12,5))
sns.boxplot(y=df['WTI_Returns'])
plt.title("Boxplot of Oil Returns")
plt.savefig(os.path.join(fig_dir, "returns_boxplot.png"))
plt.close()

plt.figure(figsize=(12,6))
df['WTI_Close'].plot(title="WTI Crude Oil Price (Monthly)")
plt.savefig(os.path.join(fig_dir, "wti_time_series.png"))
plt.close()

plt.figure(figsize=(12,6))
df['WTI_Returns'].plot(title="Oil Monthly Returns")
plt.savefig(os.path.join(fig_dir, "returns_time_series.png"))
plt.close()

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.savefig(os.path.join(fig_dir, "correlation_heatmap.png"))
plt.close()

print("\nKey Statistics:")
print(df['WTI_Returns'].describe())
print("\nLag-1 Autocorrelation of Returns:", df['WTI_Returns'].autocorr(lag=1))

Loaded columns: ['WTI_Close', 'CPIAUCSL', 'FEDFUNDS', 'INDPRO', 'DCOILWTICO']
            WTI_Close  CPIAUCSL  FEDFUNDS   INDPRO  DCOILWTICO
2010-01-31  73.889999   217.488      0.11  89.3426       72.85
2010-02-28  78.330002   217.281      0.13  89.6779       79.72
2010-03-31  82.500000   217.353      0.16  90.2928       83.45
2010-04-30  85.580002   217.403      0.20  90.5991       86.07
2010-05-31  74.900002   217.290      0.20  91.8230       74.00

Key Statistics:
count    198.000000
mean       0.816871
std       13.391101
min      -56.485268
25%       -5.514131
50%        0.398526
75%        5.981320
max      115.345266
Name: WTI_Returns, dtype: float64

Lag-1 Autocorrelation of Returns: 0.08462552563994298
